# ICSE — Geographic Diversity Time Series

Regions are read directly from the clean JSON `region` field.
To change region definitions, update `src/ingest/add_regions.py` and re-run it.

## Imports & config

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

INPUT_FILE = Path("../../data/clean/icse_with_regions.json")
OUTPUT_DIR = Path("figures/isce")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TITLE_SUFFIX = "ICSE (2010–2024)"
FIG_PREFIX   = "icse"

# ── Colors per region — add entries here if new regions appear in the data ─────
REGION_COLORS = {
    "W. Europe":            "#3266ad",
    "North America":        "#e07b3a",
    "Asia-Pacific":         "#3aa66b",
    "E. Europe":            "#7b52ab",
    "Latin America":        "#b04db0",
    "Middle East & Africa": "#c0474a",
    "Other":                "#888780",
}

# ── Country display names — used for annotations only ─────────────────────────
COUNTRY_NAMES = {
    "AU": "Australia",    "AT": "Austria",      "BE": "Belgium",
    "BR": "Brazil",       "CA": "Canada",       "CL": "Chile",
    "CN": "China",        "CZ": "Czech Rep.",   "DE": "Germany",
    "DK": "Denmark",      "DZ": "Algeria",      "ES": "Spain",
    "FI": "Finland",      "FR": "France",       "GB": "United Kingdom",
    "GR": "Greece",       "HR": "Croatia",      "HU": "Hungary",
    "IE": "Ireland",      "IN": "India",        "IS": "Iceland",
    "IT": "Italy",        "JO": "Jordan",       "LU": "Luxembourg",
    "MX": "Mexico",       "NL": "Netherlands",  "NO": "Norway",
    "NZ": "New Zealand",  "PL": "Poland",       "PT": "Portugal",
    "RO": "Romania",      "SE": "Sweden",       "SG": "Singapore",
    "SI": "Slovenia",     "TH": "Thailand",     "TN": "Tunisia",
    "TR": "Turkey",       "US": "United States","CH": "Switzerland",
    "AR": "Argentina",    "KR": "South Korea",  "JP": "Japan",
    "HK": "Hong Kong",    "TW": "Taiwan",       "MY": "Malaysia",
    "ID": "Indonesia",    "PK": "Pakistan",     "VN": "Vietnam",
    "ZA": "South Africa", "EG": "Egypt",        "IL": "Israel",
    "RS": "Serbia",       "SK": "Slovakia",     "BG": "Bulgaria",
    "RU": "Russia",       "EE": "Estonia",      "MK": "N. Macedonia",
    "CO": "Colombia",     "UY": "Uruguay",      "PE": "Peru",
    "BB": "Barbados",     "BM": "Bermuda",      "GL": "Greenland",
    "EC": "Ecuador",      "MO": "Macao",        "LY": "Libya",
    "KW": "Kuwait",       "PS": "Palestine",    "MA": "Morocco",
    "NA": "Namibia",      "AE": "UAE",          "SA": "Saudi Arabia",
    "IR": "Iran",
}

# Asia-Pacific country breakdown config
APAC_CODES_FULL      = ["CN", "JP", "AU", "KR", "SG", "HK", "TW", "IN", "NZ",
                         "TH", "MY", "ID", "PK", "VN", "MO"]
APAC_CODES_ASIA_ONLY = ["CN", "JP", "KR", "SG", "HK", "TW", "IN",
                          "TH", "MY", "ID", "PK", "VN", "MO"]

COUNTRY_COLORS = {
    "CN": "#e07b3a", "JP": "#3266ad", "AU": "#3aa66b", "KR": "#b04db0",
    "SG": "#c0474a", "HK": "#c0a030", "TW": "#5588cc", "IN": "#e84393",
    "NZ": "#7b52ab", "TH": "#888780", "MY": "#00838f", "ID": "#a05000",
    "PK": "#4caf50", "VN": "#ff5722", "MO": "#607d8b",
}

## Load data

In [ ]:
with open(INPUT_FILE) as f:
    df = (
        pd.DataFrame(json.load(f))
        .dropna(subset=["country_code"])
        .drop_duplicates(subset=["doi", "year", "country_code"])
        .copy()
    )

if "region" not in df.columns:
    raise ValueError("No 'region' column found. Run src/ingest/add_regions.py first.")

df["region"] = df["region"].fillna("Other")

# derive region order from data, using REGION_COLORS order where possible
data_regions  = df["region"].unique().tolist()
REGION_ORDER  = [r for r in REGION_COLORS if r in data_regions]
REGION_ORDER += [r for r in data_regions if r not in REGION_COLORS]

years = sorted(df["year"].unique())

print(f"Loaded {len(df):,} deduplicated rows")
print(f"Years:   {years}")
print(f"Countries: {df.country_code.nunique()}")
print(f"Regions: {REGION_ORDER}")

## Helper functions

In [ ]:
def get_region_pct(df):
    """Percentage of papers from each region per year.
    Reads region directly from the dataframe — no mapping needed.
    A paper counts once per region it has authors from.
    """
    doi_region  = df.drop_duplicates(subset=["doi", "year", "region"])
    region_year = (
        doi_region.groupby(["year", "region"]).size()
        .unstack(fill_value=0)
        .reindex(columns=REGION_ORDER, fill_value=0)
    )
    year_totals = doi_region.drop_duplicates(subset=["doi", "year"]).groupby("year").size()
    return region_year.div(year_totals, axis=0) * 100


def compute_diversity(df):
    """Shannon H, H_max, evenness, and unique country count per year."""
    rows = []
    for year, g in df.groupby("year"):
        cc_counts = (
            g.drop_duplicates(subset=["doi", "country_code"])
             .groupby("country_code")["doi"].nunique()
        )
        n = len(cc_counts)
        p = cc_counts / cc_counts.sum()
        H = float(-(p * np.log(p)).sum())
        H_max = np.log(n) if n > 1 else 1
        rows.append({
            "year":        year,
            "shannon_H":   round(H, 3),
            "H_max":       round(H_max, 3),
            "evenness":    round(H / H_max, 3),
            "n_countries": n,
            "n_papers":    g["doi"].nunique(),
        })
    return pd.DataFrame(rows).set_index("year")


def compute_hhi(df):
    """Herfindahl-Hirschman Index per year. Higher = more concentrated."""
    rows = []
    for year, g in df.groupby("year"):
        p = (
            g.drop_duplicates(subset=["doi", "country_code"])
             .groupby("country_code")["doi"].nunique()
        )
        p = p / p.sum()
        rows.append({"year": year, "hhi": round(float((p ** 2).sum()), 4)})
    return pd.DataFrame(rows).set_index("year")


def style_ax(ax, years=None):
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel("Year", labelpad=6)
    if years is not None:
        ax.set_xticks(years)


def save_fig(fig, filename):
    fig.savefig(OUTPUT_DIR / filename, dpi=150, bbox_inches="tight")
    print(f"Saved → {filename}")


# precompute once
region_pct = get_region_pct(df)
div_df     = compute_diversity(df)
hhi_df     = compute_hhi(df)

print("Precomputed region_pct, div_df, hhi_df")
print(div_df)

## Regional share over time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for region in REGION_ORDER:
    if region not in region_pct.columns:
        continue
    if region_pct[region].max() < 1:
        continue
    ax.plot(years, region_pct[region], "o-",
            color=REGION_COLORS.get(region, "#333"),
            linewidth=2, markersize=5, label=region)

ax.set_ylabel("Share of contributions (%)", labelpad=6)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_ylim(bottom=0)
ax.legend(loc="upper right", fontsize=9, framealpha=0.9)
ax.set_title(f"Regional share of contributions per year\n{TITLE_SUFFIX}", pad=10)
style_ax(ax, years)
plt.tight_layout()
save_fig(fig, f"{FIG_PREFIX}_fig_region_line.png")
plt.show()

## Shannon H and unique countries

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4.5))
ax2 = ax1.twinx()

l1, = ax1.plot(div_df.index, div_df["shannon_H"], "o-",
               color="#3266ad", lw=2, ms=5, label="Shannon H (left)")
l2, = ax2.plot(div_df.index, div_df["n_countries"], "s--",
               color="#e07b3a", lw=2, ms=5, label="Unique countries (right)")

ax1.set_ylabel("Shannon diversity index H", color="#3266ad", labelpad=6)
ax2.set_ylabel("Unique countries",          color="#e07b3a", labelpad=6)
ax1.tick_params(axis="y", labelcolor="#3266ad")
ax2.tick_params(axis="y", labelcolor="#e07b3a")
ax1.set_ylim(bottom=0)
ax1.spines[["top"]].set_visible(False)
ax2.spines[["top"]].set_visible(False)
ax1.legend([l1, l2], [l1.get_label(), l2.get_label()],
           loc="lower left", fontsize=9, framealpha=0.9)
ax1.set_title(f"Geographic diversity over time\n{TITLE_SUFFIX}", pad=10)
style_ax(ax1, div_df.index.tolist())
plt.tight_layout()
save_fig(fig, f"{FIG_PREFIX}_fig_diversity.png")
plt.show()

## Shannon H and evenness

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4.5))
ax2 = ax1.twinx()

l1, = ax1.plot(div_df.index, div_df["shannon_H"], "o-",
               color="#e07b3a", lw=2, ms=5, label="Shannon H (left)")
l2, = ax1.plot(div_df.index, div_df["H_max"], "o--",
               color="#e07b3a", lw=1, ms=4, alpha=0.4, label="H_max (left)")
l3, = ax2.plot(div_df.index, div_df["evenness"], "s-",
               color="#3266ad", lw=2, ms=5, label="Evenness H/H_max (right)")

ax1.set_ylabel("Shannon diversity index H", color="#e07b3a", labelpad=6)
ax2.set_ylabel("Evenness (0–1)",             color="#3266ad", labelpad=6)
ax1.tick_params(axis="y", labelcolor="#e07b3a")
ax2.tick_params(axis="y", labelcolor="#3266ad")
ax2.set_ylim(0, 1)
ax1.set_ylim(bottom=0)
ax1.spines[["top"]].set_visible(False)
ax2.spines[["top"]].set_visible(False)
ax1.legend([l1, l2, l3], [l1.get_label(), l2.get_label(), l3.get_label()],
           loc="lower left", fontsize=9, framealpha=0.9)
ax1.set_title(f"Shannon diversity and evenness over time\n{TITLE_SUFFIX}", pad=10)
style_ax(ax1, div_df.index.tolist())
plt.tight_layout()
save_fig(fig, f"{FIG_PREFIX}_fig_evenness.png")
plt.show()

## Herfindahl-Hirschman Index (HHI)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(hhi_df.index, hhi_df["hhi"], "o-", color="#3266ad", lw=2, ms=5)
ax.axhline(0.25, color="#c0474a", lw=1, linestyle="--", alpha=0.7,
           label="Highly concentrated (0.25)")
ax.axhline(0.10, color="#3aa66b", lw=1, linestyle="--", alpha=0.7,
           label="Moderately concentrated (0.10)")
ax.set_ylabel("HHI", labelpad=6)
ax.set_ylim(bottom=0)
ax.legend(fontsize=9, framealpha=0.9)
ax.set_title(f"HHI over time — Higher = more concentrated\n{TITLE_SUFFIX}", pad=10)
style_ax(ax, hhi_df.index.tolist())
plt.tight_layout()
save_fig(fig, f"{FIG_PREFIX}_fig_hhi.png")
plt.show()

## First appearance year per country

In [ ]:
first_year = (
    df.dropna(subset=["country_code"])
    .groupby("country_code")["year"].min()
    .reset_index()
    .rename(columns={"year": "first_year"})
)
first_year["country_name"] = first_year["country_code"].map(
    lambda x: COUNTRY_NAMES.get(x, x)
)

debut_counts = first_year.groupby("first_year").size().reset_index(name="n_new")
debut_names  = first_year.groupby("first_year")["country_name"].apply(list)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(debut_counts["first_year"], debut_counts["n_new"],
       color="#3266ad", edgecolor="white", linewidth=0.4)

for yr, names in debut_names.items():
    label = "\n".join(names[:6])
    if len(names) > 6:
        label += f"\n+{len(names)-6} more"
    ax.text(yr, len(names) + 0.1, label,
            ha="center", va="bottom", fontsize=6.5, color="#333")

ax.set_ylabel("Number of new countries", labelpad=6)
ax.set_ylim(bottom=0)
ax.set_title(f"First appearance year per country\n{TITLE_SUFFIX}", pad=10)
style_ax(ax, years)
plt.tight_layout()
save_fig(fig, f"{FIG_PREFIX}_fig_first_appearance.png")
plt.show()

## Asia-Pacific country breakdown

In [ ]:
year_totals = df.drop_duplicates(subset=["doi", "year"]).groupby("year")["doi"].nunique()

def plot_apac_breakdown(country_codes, subtitle, filename):
    apac_counts = (
        df[df["country_code"].isin(country_codes)]
        .drop_duplicates(subset=["doi", "country_code"])
        .groupby(["year", "country_code"])["doi"].nunique()
        .unstack(fill_value=0)
        .reindex(years, fill_value=0)   # fill gaps so lines connect
    )
    col_order = apac_counts.sum().sort_values(ascending=False).index.tolist()
    apac_pct  = apac_counts[col_order].div(year_totals, axis=0) * 100

    fig, ax = plt.subplots(figsize=(12, 5))
    for cc in col_order:
        if apac_pct[cc].max() < 0.5:
            continue
        ax.plot(apac_pct.index, apac_pct[cc], "o-",
                color=COUNTRY_COLORS.get(cc, "#333"),
                lw=2, ms=4, label=COUNTRY_NAMES.get(cc, cc))

    ax.set_ylabel("Share of total contributions (%)", labelpad=6)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylim(bottom=0)
    ax.legend(loc="upper left", fontsize=8, framealpha=0.9, ncol=2)
    ax.set_title(f"Asia-Pacific breakdown — {subtitle}\n{TITLE_SUFFIX}", pad=10)
    style_ax(ax, years)
    plt.tight_layout()
    save_fig(fig, filename)
    plt.show()

plot_apac_breakdown(APAC_CODES_FULL,
                    "Including Australia & New Zealand",
                    f"{FIG_PREFIX}_fig_apac_full.png")

plot_apac_breakdown(APAC_CODES_ASIA_ONLY,
                    "Asia only (excluding Australia & New Zealand)",
                    f"{FIG_PREFIX}_fig_apac_asia_only.png")